> First time use: follow instructions in the README.md file in this directory.

# Network analysis of the Jesuit Mission in China

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
tlnb.print_info()


In [ ]:
# generate network of those on the same voyage
from timelink.networks.network_generation import network_from_attribute
from timelink.pandas import attribute_values

attribute = 'wicky-viagem'  # attribute to use for the network

attribute_values_df = attribute_values(attribute,db=tlnb.db)
attribute_values_df

In [ ]:
wg = network_from_attribute(attribute='wicky-viagem',mode='value-node',db=tlnb.db)
[(m,wg.nodes[m]) for m in list(wg.nodes)[0:3]]

In [ ]:
from networkx.drawing.nx_agraph import write_dot

write_dot(wg,'wicki-viagens.dot')

In [ ]:
# generate network of those on the same college
jc = network_from_attribute('jesuita-entrada', mode='value-node',db=tlnb.db)
write_dot(jc,'jesuitas-entrada-colegios.dot')

In [ ]:
[jc.nodes[m] for m in list(jc.nodes)[0:6]]

## Visualizar

In [ ]:

from timelink.networks.network_draw import draw_network

# Ensure that the graph is in G
G=wg

#Choose a title!
title = 'jesuit_travelling_network'
tooltips = [("desc","@desc"),("type","@type"), ("date","@date")]

node_color_attr = 'type'
node_types = {G.nodes[n][node_color_attr] for n in G.nodes}
print(f"Node types: {node_types}")
node_colors = {'person':'green', 'wicky-viagem':'red'}

draw_network(G,
            title=title,
            node_colors=node_colors,
            node_tooltips=tooltips,
            iterations=100,
            width=800,
            height=800,)




In [ ]:
from timelink.networks import draw_network
# Ensure that the graph is in G
G=jc

#Choose a title!
title = 'jesuit_entry_college'

tooltips = [("desc","@desc"),
            ("type","@type"),
            ("original_id","@original_id"),
            ("date","@date")]

node_color_attr = 'type'
# find nodes with no type attribute
for n in G.nodes:
    if 'type' not in G.nodes[n]:
        print(f"Node {n} has no type attribute: {G.nodes[n]}")
node_types = {G.nodes[n][node_color_attr] for n in G.nodes}
print(f"Node types: {node_types}")
node_colors = {'person':'green', 'jesuita-entrada':'blue'}


draw_network(G,
            title=title,
            node_color_attr=node_color_attr,
            node_colors=node_colors,
            node_tooltips=tooltips,
            width=800,
            height=800)



## Join both networks

In [ ]:
import networkx as nx
from networkx import MultiGraph

jesuit_net = MultiGraph()

jesuit_net=nx.compose(wg,jc)

write_dot(jesuit_net,'college_and_voyage.dot')

In [ ]:
#Choose a title!
title = 'college_and_voyage'

# Ensure that the graph is in G
G=jesuit_net

tooltips = [("desc","@desc"),
            ("type","@type"),
            ("original_id","@id"),
            ("date", "@date")]
node_types = {G.nodes[n][node_color_attr] for n in G.nodes}
print(f"Node types: {node_types}")
node_colors = {'person':'green', 'jesuita-entrada':'blue', 'wicky-viagem':'red'}
draw_network(G,
            title=title,
            node_color_attr='type',
            node_colors=node_colors,
            node_tooltips=tooltips,
            iterations=100,
            height=800,  # Adjust height and width as needed
            width=800,)




## Places of stay

In [ ]:
from timelink.pandas import attribute_values

places_of_stay = attribute_values('estadia@wikidata',db=tlnb.db)
places_of_stay

In [ ]:
import requests
import time

estadias = network_from_attribute('estadia@wikidata',mode='value-node',db=tlnb.db)
for node in estadias.nodes:
    if estadias.nodes[node].get('type') == 'estadia@wikidata' and 'id' in estadias.nodes[node]:
        wikidata_url = estadias.nodes[node]['id']
        wikidata_id = wikidata_url.split('/')[-1]
        time.sleep(0.5)
        response = requests.get(f"https://www.wikidata.org/wiki/Special:EntityData/{wikidata_id}.json")
        if response.status_code == 200:
            data = response.json()
            if 'entities' in data and wikidata_id in data['entities']:
                entity = data['entities'][wikidata_id]
                if 'labels' in entity and 'en' in entity['labels']:
                    estadias.nodes[node]['desc'] = entity['labels']['en']['value']
                    print(f"Node {node} updated with description: {estadias.nodes[node]['desc']}")
[estadias.nodes[m] for m in list(estadias.nodes)[0:3]]

In [ ]:
# visualize
title = 'place of stay'

# Ensure that the graph is in G
G=estadias

tooltips = [("desc","@desc"),
            ("type","@type"),
            ("original_id","@id"),
            ("date", "@date")]
node_types = {G.nodes[n][node_color_attr] for n in G.nodes}
print(f"Node types: {node_types}")
node_colors = {'person':'green', 'jesuita-entrada':'blue', 'wicky-viagem':'red', 'estadia@wikidata':'yellow'}
draw_network(G,
            title=title,
            node_color_attr='type',
            node_colors=node_colors,
            node_tooltips=tooltips,
            iterations=300,
            height=800,  # Adjust height and width as needed
            width=800,)
